# Story 1.1: Agent Integration Test

Simple test to verify:
1. Scenario Generator Agent creates scenarios
2. Simulation Feedback Agent provides feedback
3. MLflow traces are logged to summit-sim experiment

## 1. Setup & Imports

In [1]:
import mlflow

from summit_sim.agents.generator import generate_scenario
from summit_sim.agents.simulation import process_choice
from summit_sim.schemas import HostConfig
from summit_sim.settings import settings

# Initialize MLflow
mlflow.set_tracking_uri(settings.mlflow_tracking_uri)
mlflow.set_experiment(settings.mlflow_experiment_name)
mlflow.pydantic_ai.autolog()

print(f"✓ MLflow tracking URI: {settings.mlflow_tracking_uri}")
print(f"✓ Experiment name: {settings.mlflow_experiment_name}")
print(f"✓ Model: {settings.default_model}")
print(f"✓ API Key configured: {bool(settings.openrouter_api_key)}")

✓ MLflow tracking URI: https://mlflow.bhamm-lab.com
✓ Experiment name: summit-sim
✓ Model: google/gemini-3.1-flash-lite-preview
✓ API Key configured: True


## 2. Test Scenario Generator

In [2]:
# Test configuration
config = HostConfig(num_participants=3, activity_type="hiking", difficulty="med")

print(f"Generating scenario: {config}")
print("-" * 60)

# Generate scenario
scenario = await generate_scenario(config)

print(f"✓ Title: {scenario.title}")
print(f"✓ Setting: {scenario.setting}")
print(f"✓ Patient: {scenario.patient_summary}")
print(f"✓ Turns: {len(scenario.turns)}")
print(f"✓ Starting turn: {scenario.starting_turn_id}")
print("\nLearning Objectives:")
for obj in scenario.learning_objectives:
    print(f"  • {obj}")

Generating scenario: num_participants=3 activity_type='hiking' difficulty='med'
------------------------------------------------------------
✓ Title: Alpine Emergency: The Slow Descent
✓ Setting: Alpine hiking trail, elevation 11,000 ft. Mid-afternoon. Weather is clear but breezy. Trailhead is 5 miles away (downhill).
✓ Patient: Alex, 35 years old, experienced hiker, no known medical history. Currently hiking at 11,000 feet elevation. Persistent headache, nausea, mild ataxia.
✓ Turns: 5
✓ Starting turn: initial_encounter

Learning Objectives:
  • Recognize symptoms of progressive altitude illness beyond simple fatigue or dehydration.
  • Identify early behavioral/physical signs of High Altitude Cerebral Edema (HACE).
  • Prioritize immediate descent as the definitive treatment for altitude illness.


Trace(trace_id=tr-e6c824e3a7ff6be1e4966f6f1ff60505)

## 3. Test Simulation Feedback (Single Turn)

In [3]:
# Get starting turn
current_turn = scenario.get_turn(scenario.starting_turn_id)

print(f"Turn {current_turn.turn_id}:")
print(f"{current_turn.narrative_text}")
print("\nChoices:")
for i, choice in enumerate(current_turn.choices):
    print(f"  {i + 1}. {choice.description}")

# Select first choice
selected_choice = current_turn.choices[0]
print(f"\n>>> Selecting: {selected_choice.description}")
print("-" * 60)

# Get feedback
result = await process_choice(scenario, current_turn, selected_choice)

print("\nFeedback:")
print(result.feedback)
print("\nLearning Moments:")
for moment in result.learning_moments:
    print(f"  • {moment}")
print(f"\n✓ Scenario complete: {result.is_complete}")

Turn initial_encounter:
You are a three-person hiking group. Your friend, Alex, slows down significantly on a steep switchback. Alex looks pale, is holding their head, and states, 'I've got a killer headache and I feel like I'm going to throw up.' They stumble slightly as they try to take a step. What do you do?

Choices:
  1. Have Alex sit down, drink plenty of water, and eat a snack to recover energy.
  2. Conduct a thorough patient assessment, checking LOC, balance, and headache severity.

>>> Selecting: Have Alex sit down, drink plenty of water, and eat a snack to recover energy.
------------------------------------------------------------

Feedback:
While hydration and rest are important in the wilderness, assuming they are the sole solution for someone presenting with a 'killer headache' and ataxia at 11,000 feet can be dangerous. These symptoms suggest a serious neurological condition rather than simple fatigue, and by stopping for a snack, we may be delaying an urgent descent t

Trace(trace_id=tr-668e2a3a2bacdb2b83fb8542d5c56a6e)

## 4. Test Multiple Activities (Optional)

In [4]:
# Quick test of all activities
activities = ["canyoneering", "skiing", "hiking"]

for activity in activities:
    config = HostConfig(num_participants=2, activity_type=activity, difficulty="low")
    test_scenario = await generate_scenario(config)
    print(f"✓ {activity}: {test_scenario.title[:50]}...")

✓ canyoneering: The Heat in the Slot...
✓ skiing: Whispering Pines Ski Incident...
✓ hiking: Mid-Day Slump on the Ridge...


[Trace(trace_id=tr-71eddd1ac13b0cd8aa3987bd9a855b7c), Trace(trace_id=tr-d467d000999f2bd0a7eb7aa371436dd8), Trace(trace_id=tr-d72346ba710d4da5bb0b5a710d12ca64)]

## 5. MLflow Verification

Check your MLflow UI at the tracking URI to verify:
- Traces appear under the `summit-sim` experiment
- Each agent call is logged with latency metrics
- Token usage is tracked

In [5]:
# Show experiment info
experiment = mlflow.get_experiment_by_name(settings.mlflow_experiment_name)
print(f"Experiment ID: {experiment.experiment_id}")
print(f"Artifact Location: {experiment.artifact_location}")
print(f"\nView traces at: {settings.mlflow_tracking_uri}")

Experiment ID: 7
Artifact Location: mlflow-artifacts:/7

View traces at: https://mlflow.bhamm-lab.com


---
## ✅ Integration Test Complete

If you see checkmarks and no errors, Story 1.1 is working!